# 🏦 Loan Approval Classification

**Dataset**: [Loan Approval Classification Dataset](https://www.kaggle.com/datasets/taweilo/loan-approval-classification-data)

- **45,000** records | **14** features | Binary target: `loan_status` (1=Approved, 0=Rejected)
- Features: Person demographics, Loan details, Credit history

**Approach**: EDA → Preprocessing → Train multiple ML models → Compare accuracy → Save best model

## Step 1: Setup & Install Dependencies

In [ ]:
!pip install -q kaggle xgboost

## Step 2: Download Dataset from Kaggle

Upload your `kaggle.json` API key. Get it from: [kaggle.com/settings](https://www.kaggle.com/settings) → **Create New Token**

In [ ]:
import os
from google.colab import files

uploaded = files.upload()
os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('✅ Kaggle API key configured!')

In [ ]:
!kaggle datasets download -d taweilo/loan-approval-classification-data --unzip -p /content/dataset
print('\n✅ Dataset downloaded and extracted!')

## Step 3: Load & Explore the Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load dataset
csv_files = [f for f in os.listdir('/content/dataset') if f.endswith('.csv')]
print(f'📂 Files: {csv_files}')
df = pd.read_csv(f'/content/dataset/{csv_files[0]}')

print(f'\n📊 Dataset Shape: {df.shape}')
print(f'   Rows: {df.shape[0]:,} | Columns: {df.shape[1]}')
print(f'\n📋 Columns:')
for i, col in enumerate(df.columns, 1):
    print(f'   {i:2d}. {col} ({df[col].dtype})')

df.head()

In [ ]:
# Dataset info & statistics
print('📊 Statistical Summary:')
print('=' * 60)
df.describe()

In [ ]:
# Missing values check
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing': missing, '%': missing_pct})
missing_df = missing_df[missing_df['Missing'] > 0].sort_values('Missing', ascending=False)

if len(missing_df) > 0:
    print('⚠️ Missing Values:')
    print(missing_df)
else:
    print('✅ No missing values!')

# Check for outliers (e.g., age > 100)
if 'person_age' in df.columns:
    outlier_age = df[df['person_age'] > 100]
    print(f'\n⚠️ Age > 100 years: {len(outlier_age)} records (will be removed)')

## Step 4: Exploratory Data Analysis (EDA)

In [ ]:
# Target variable distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

target_counts = df['loan_status'].value_counts()
colors = ['#e74c3c', '#2ecc71']
labels = ['Rejected (0)', 'Approved (1)']

# Bar chart
axes[0].bar(labels, target_counts.sort_index().values, color=colors, edgecolor='black')
for i, (count, pct) in enumerate(zip(target_counts.sort_index().values,
                                      target_counts.sort_index().values / len(df) * 100)):
    axes[0].text(i, count + 300, f'{count:,}\n({pct:.1f}%)', ha='center', fontweight='bold')
axes[0].set_title('🎯 Loan Status Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')

# Pie chart
axes[1].pie(target_counts.sort_index().values, labels=labels, autopct='%1.1f%%',
            colors=colors, explode=[0.05, 0], shadow=True, startangle=90, textprops={'fontsize': 12})
axes[1].set_title('🎯 Approval Ratio', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Numerical features distributions
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
num_features = [c for c in numerical_cols if c != 'loan_status']

n_cols = 4
n_rows = (len(num_features) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(num_features):
    # Plot by class
    df[df['loan_status'] == 0][col].hist(bins=30, ax=axes[i], color='#e74c3c', alpha=0.5, label='Rejected')
    df[df['loan_status'] == 1][col].hist(bins=30, ax=axes[i], color='#2ecc71', alpha=0.5, label='Approved')
    axes[i].set_title(col, fontweight='bold')
    axes[i].legend(fontsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('📊 Feature Distributions by Loan Status', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Categorical features analysis
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f'📋 Categorical columns: {categorical_cols}')

if len(categorical_cols) > 0:
    n_cats = min(len(categorical_cols), 6)
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()

    for i, col in enumerate(categorical_cols[:6]):
        approval_rate = df.groupby(col)['loan_status'].mean().sort_values(ascending=False)
        approval_rate.plot(kind='bar', ax=axes[i], color='#3498db', edgecolor='black', alpha=0.8)
        axes[i].set_title(f'Approval Rate by {col}', fontweight='bold')
        axes[i].set_ylabel('Approval Rate')
        axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=45, ha='right')
        axes[i].axhline(y=df['loan_status'].mean(), color='red', linestyle='--', alpha=0.7)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle('📊 Loan Approval Rate by Category', fontsize=16, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(14, 10))
corr_matrix = df[numerical_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, square=True)
plt.title('🔗 Feature Correlation Heatmap', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Top correlations with target
print('🎯 Top Features Correlated with Loan Status:')
target_corr = corr_matrix['loan_status'].drop('loan_status').abs().sort_values(ascending=False)
for feat, val in target_corr.head(10).items():
    direction = '↑' if corr_matrix.loc[feat, 'loan_status'] > 0 else '↓'
    print(f'   {direction} {feat}: {corr_matrix.loc[feat, "loan_status"]:.4f}')

## Step 5: Data Preprocessing

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

data = df.copy()

# Remove outliers (age > 100)
if 'person_age' in data.columns:
    before = len(data)
    data = data[data['person_age'] <= 100]
    print(f'✅ Removed {before - len(data)} outlier records (age > 100)')

# Handle missing values
for col in data.columns:
    if data[col].isnull().sum() > 0:
        if data[col].dtype in ['float64', 'int64']:
            data[col].fillna(data[col].median(), inplace=True)
        else:
            data[col].fillna(data[col].mode()[0], inplace=True)

print(f'✅ Missing values handled. Remaining: {data.isnull().sum().sum()}')

# Encode categorical variables
label_encoders = {}
cat_cols = data.select_dtypes(include=['object']).columns.tolist()

for col in cat_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col].astype(str))
    label_encoders[col] = le
    print(f'   Encoded {col}: {list(le.classes_)}')

# Separate features and target
X = data.drop('loan_status', axis=1)
y = data['loan_status']

# Train/Test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'\n📊 Data Split:')
print(f'   Training: {X_train.shape[0]:,} samples ({X_train.shape[1]} features)')
print(f'   Testing:  {X_test.shape[0]:,} samples')
print(f'   Target:   Approved={y_train.sum():,} | Rejected={len(y_train)-y_train.sum():,}')

## Step 6: Train Multiple Models

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, roc_auc_score, f1_score,
                              precision_score, recall_score)
import time

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=10),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=200, random_state=42, use_label_encoder=False,
                              eval_metric='logloss', verbosity=0),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=7),
}

results = {}
print('🚀 Training Models...\n')
print(f'{"Model":<25} {"Accuracy":>10} {"F1-Score":>10} {"AUC-ROC":>10} {"Time":>8}')
print('=' * 66)

for name, model in models.items():
    start = time.time()

    if name in ['Logistic Regression', 'KNN']:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]

    elapsed = time.time() - start
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    results[name] = {
        'model': model, 'accuracy': acc, 'f1_score': f1,
        'auc_roc': auc, 'y_pred': y_pred, 'y_prob': y_prob,
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'time': elapsed
    }

    print(f'{name:<25} {acc:>9.4f} {f1:>10.4f} {auc:>10.4f} {elapsed:>7.2f}s')

best_model_name = max(results, key=lambda x: results[x]['accuracy'])
print(f'\n🏆 Best Model: {best_model_name} (Accuracy: {results[best_model_name]["accuracy"]*100:.2f}%)')

## Step 7: Model Comparison Charts

In [ ]:
model_names = list(results.keys())
metrics_data = {
    'Accuracy': [results[m]['accuracy'] * 100 for m in model_names],
    'F1-Score': [results[m]['f1_score'] * 100 for m in model_names],
    'AUC-ROC': [results[m]['auc_roc'] * 100 for m in model_names],
    'Precision': [results[m]['precision'] * 100 for m in model_names],
    'Recall': [results[m]['recall'] * 100 for m in model_names],
}

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
colors_bar = ['#2ecc71' if m == best_model_name else '#3498db' for m in model_names]

for i, (metric, values) in enumerate(list(metrics_data.items())[:3]):
    bars = axes[i].barh(model_names, values, color=colors_bar, edgecolor='black', linewidth=0.5)
    for bar, val in zip(bars, values):
        axes[i].text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2.,
                     f'{val:.2f}%', va='center', fontweight='bold', fontsize=9)
    axes[i].set_xlabel(f'{metric} (%)')
    axes[i].set_title(f'🎯 {metric}', fontsize=13, fontweight='bold')
    axes[i].set_xlim(0, max(values) + 8)

plt.suptitle('📊 Model Comparison', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Step 8: Best Model - Detailed Evaluation

In [ ]:
best = results[best_model_name]

print(f'🏆 Best Model: {best_model_name}')
print('=' * 50)
print(f'   Accuracy:  {best["accuracy"]*100:.2f}%')
print(f'   F1-Score:  {best["f1_score"]*100:.2f}%')
print(f'   AUC-ROC:   {best["auc_roc"]*100:.2f}%')
print(f'   Precision: {best["precision"]*100:.2f}%')
print(f'   Recall:    {best["recall"]*100:.2f}%')

print('\n📋 Classification Report:\n')
print(classification_report(y_test, best['y_pred'], target_names=['Rejected', 'Approved']))

In [ ]:
# Confusion Matrices
cm = confusion_matrix(y_test, best['y_pred'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Rejected', 'Approved'], yticklabels=['Rejected', 'Approved'],
            linewidths=0.5)
axes[0].set_title(f'🔍 Confusion Matrix - {best_model_name}', fontweight='bold')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')

cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='YlOrRd', ax=axes[1],
            xticklabels=['Rejected', 'Approved'], yticklabels=['Rejected', 'Approved'],
            linewidths=0.5)
axes[1].set_title('🔍 Normalized Confusion Matrix', fontweight='bold')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

## Step 9: ROC Curves

In [ ]:
from sklearn.metrics import roc_curve

plt.figure(figsize=(10, 7))
colors_roc = plt.cm.Set1(np.linspace(0, 1, len(results)))

for (name, res), color in zip(results.items(), colors_roc):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    lw = 3 if name == best_model_name else 1.5
    style = '-' if name == best_model_name else '--'
    plt.plot(fpr, tpr, color=color, lw=lw, linestyle=style,
             label=f'{name} (AUC = {res["auc_roc"]:.4f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Random (AUC = 0.5)')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('📈 ROC Curves - All Models', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 10: Feature Importance

In [ ]:
# Feature importance from best tree-based model
tree_models = ['Random Forest', 'XGBoost', 'Gradient Boosting', 'Decision Tree']
fi_name = best_model_name if best_model_name in tree_models else 'Random Forest'
fi_model = results[fi_name]['model']

importances = fi_model.feature_importances_
feat_imp = pd.DataFrame({'Feature': X.columns, 'Importance': importances})
feat_imp = feat_imp.sort_values('Importance', ascending=True)

plt.figure(figsize=(10, 8))
colors_fi = plt.cm.viridis(np.linspace(0.2, 0.8, len(feat_imp)))
bars = plt.barh(feat_imp['Feature'], feat_imp['Importance'], color=colors_fi, edgecolor='black', linewidth=0.5)

for bar, val in zip(bars, feat_imp['Importance']):
    plt.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2.,
             f'{val:.4f}', va='center', fontsize=9)

plt.xlabel('Importance', fontsize=12)
plt.title(f'🔑 Feature Importance ({fi_name})', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n🔝 Top 5 Most Important Features for Loan Approval:')
for _, row in feat_imp.tail(5).iloc[::-1].iterrows():
    print(f'   • {row["Feature"]}: {row["Importance"]:.4f}')

## Step 11: Summary

In [ ]:
print('\n' + '=' * 60)
print('📋 FINAL SUMMARY')
print('=' * 60)
print(f'  Dataset:         Loan Approval Classification')
print(f'  Total Samples:   {len(data):,}')
print(f'  Features:        {X.shape[1]}')
print(f'  Target:          loan_status (Approved/Rejected)')
print(f'  Train Samples:   {len(X_train):,}')
print(f'  Test Samples:    {len(X_test):,}')
print(f'\n  🏆 Best Model:   {best_model_name}')
print(f'  Accuracy:        {results[best_model_name]["accuracy"]*100:.2f}%')
print(f'  F1-Score:        {results[best_model_name]["f1_score"]*100:.2f}%')
print(f'  AUC-ROC:         {results[best_model_name]["auc_roc"]*100:.2f}%')
print(f'\n  📊 All Model Rankings:')
for rank, (name, res) in enumerate(
    sorted(results.items(), key=lambda x: x[1]['accuracy'], reverse=True), 1):
    medal = ['🥇', '🥈', '🥉'][rank-1] if rank <= 3 else '  '
    print(f'  {medal} #{rank} {name:<25} Acc: {res["accuracy"]*100:.2f}%  '
          f'F1: {res["f1_score"]*100:.2f}%  AUC: {res["auc_roc"]*100:.2f}%')
print('=' * 60)

## Step 12: Save the Best Model

In [ ]:
import joblib
import json

os.makedirs('/content/saved_model', exist_ok=True)

# Save best model
model_path = f'/content/saved_model/loan_approval_model.pkl'
joblib.dump(results[best_model_name]['model'], model_path)
print(f'✅ Model saved: {model_path}')

# Save scaler
joblib.dump(scaler, '/content/saved_model/scaler.pkl')
print(f'✅ Scaler saved')

# Save label encoders
joblib.dump(label_encoders, '/content/saved_model/label_encoders.pkl')
print(f'✅ Label encoders saved')

# Save model info
model_info = {
    'best_model': best_model_name,
    'accuracy': results[best_model_name]['accuracy'],
    'f1_score': results[best_model_name]['f1_score'],
    'auc_roc': results[best_model_name]['auc_roc'],
    'features': list(X.columns),
    'categorical_columns': cat_cols,
    'all_results': {name: {'accuracy': res['accuracy'], 'f1': res['f1_score'], 'auc': res['auc_roc']}
                    for name, res in results.items()}
}
with open('/content/saved_model/model_info.json', 'w') as f:
    json.dump(model_info, f, indent=2)
print(f'✅ Model info saved')

# Zip and download
!cd /content && zip -r saved_model.zip saved_model/
files.download('/content/saved_model.zip')
print('\n✅ Model downloaded as saved_model.zip!')